Ссылка на урок: [Спринт 2/5: 2 RAG: от сырых данных до точных ответов → Тема 2/4: Векторные базы и индексы → Урок 5/5: Chroma и Qdrant: когда нужна полноценная векторная база]( https://practicum.yandex.ru/learn/ai-engineer/courses/99ad0dee-d9e6-4da1-b921-64e7e9653a4e/sprints/1104074/topics/7c1be2cd-4a52-4f8a-8dce-db7d2e89a79d/lessons/154daaa7-5c50-49ac-ad5d-3cac1c524c4a/#9fd67c59-bb0f-4533-a45c-b1681839331b )

In [1]:
import json
import numpy as np
from sentence_transformers import SentenceTransformer

# --- Загружаем корпус: список из 100 словарей с текстом и метаданными ---
with open("corpus.json", encoding="utf-8") as f:
    corpus = json.load(f)

texts = [c["text"] for c in corpus]
metas = [{k: c[k] for k in ("category", "source", "date", "date_int")}
         for c in corpus]


def embed_passages(model, texts):
    """Эмбеддинги документов через multilingual-e5: префикс 'passage: '."""
    inputs = ["passage: " + t for t in texts]
    return model.encode(inputs, normalize_embeddings=True, batch_size=64)


# --- Считаем эмбеддинги корпуса (e5-base, 768 координат) и сохраняем ---
e5 = SentenceTransformer("intfloat/multilingual-e5-base", device="cuda")
corpus_vecs = np.ascontiguousarray(embed_passages(e5, texts), dtype="float32")

np.save("kb_vecs.npy", corpus_vecs)   # Эмбеддинги — в файл, переиспользуем

print(f"Чанков: {len(texts)}, эмбеддинги: {corpus_vecs.shape}")
print("Пример метаданных:", metas[1])

/home/ubuntu/rag-vector-db/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Чанков: 100, эмбеддинги: (100, 768)
Пример метаданных: {'category': 'юридический', 'source': 'confluence', 'date': '2025-02-08', 'date_int': 20250208}


In [5]:
import json
import numpy as np
import chromadb
from sentence_transformers import SentenceTransformer

# --- Загружаем подготовленные эмбеддинги и корпус ---
corpus_vecs = np.load("kb_vecs.npy")
with open("corpus.json", encoding="utf-8") as f:
    corpus = json.load(f)
texts = [c["text"] for c in corpus]
metas = [{k: c[k] for k in ("category", "source", "date", "date_int")}
         for c in corpus]

# --- Создаём локальную коллекцию Chroma ---
client = chromadb.PersistentClient(path="chroma_kb")     # Хранение на диске
try:
    client.delete_collection("knowledge_base")           # Чистим прошлый прогон
except Exception:
    pass
# Пропуск 1: создайте коллекцию "knowledge_base" с косинусной метрикой
col = client.create_collection(
    name="knowledge_base",
    configuration={"hnsw": {"space": "cosine"}},  # Косинусная близость
)


# Пропуск 2: добавьте 100 записей — ids, embeddings, documents, metadatas
col.add(
    ids= [str(i) for i in range(len(texts))],
    embeddings=corpus_vecs.tolist(),
    documents=texts,
    metadatas=metas,
)
print("Записей в коллекции:", col.count())

# --- Семантический поиск с фильтром по категории ---
QUERY = "сроки и условия возврата средств"
e5 = SentenceTransformer("intfloat/multilingual-e5-base")
q_vec = e5.encode(["query: " + QUERY], normalize_embeddings=True)[0]

# Пропуск 3: поиск топ-3 с фильтром where по категории "юридический"
res = col.query(
    query_embeddings=[q_vec.tolist()],
    n_results=3,
    where={"category": "юридический"},
)

print(f"\nЗапрос: {QUERY!r} | фильтр category='юридический'")
for doc, m, dist in zip(res["documents"][0], res["metadatas"][0],
                        res["distances"][0]):
    print(f"  dist={dist:.3f} [{m['category']}/{m['source']}] {doc}")

# Для сравнения — тот же запрос без фильтра
res2 = col.query(query_embeddings=[q_vec.tolist()], n_results=3)
print("\nБез фильтра категории первых трёх:",
      [m["category"] for m in res2["metadatas"][0]])

Записей в коллекции: 100

Запрос: 'сроки и условия возврата средств' | фильтр category='юридический'
  dist=0.173 [юридический/web] Возврат предоплаты осуществляется в течение десяти рабочих дней.
  dist=0.196 [юридический/confluence] Расторжение договора возможно при письменном уведомлении за месяц.
  dist=0.202 [юридический/pdf] Гарантийные обязательства действуют двенадцать месяцев с даты продажи.

Без фильтра категории первых трёх: ['поддержка', 'юридический', 'финансы']


In [9]:
import json
import numpy as np
from qdrant_client import QdrantClient, models
from sentence_transformers import SentenceTransformer

# --- Загружаем подготовленные эмбеддинги и корпус ---
corpus_vecs = np.load("kb_vecs.npy")
with open("corpus.json", encoding="utf-8") as f:
    corpus = json.load(f)
texts = [c["text"] for c in corpus]
metas = [{k: c[k] for k in ("category", "source", "date", "date_int")}
         for c in corpus]
d = corpus_vecs.shape[1]

# --- Подключаемся к серверу Qdrant (поднят в Docker на порту 6333) ---
client = QdrantClient(host="localhost", port=6333)
if client.collection_exists("knowledge_base"):
    client.delete_collection("knowledge_base")           # Чистим прошлый прогон

# Пропуск 1: создайте коллекцию с VectorParams(size=d, distance=COSINE)
client.create_collection(
    "knowledge_base",
    vectors_config=models.VectorParams(
        size=d,
        distance=models.Distance.COSINE,
    )
)

# Индекс по полю payload ускоряет фильтрацию по категории
client.create_payload_index(
    "knowledge_base",
    field_name="category",
    field_schema=models.PayloadSchemaType.KEYWORD,
)

# Пропуск 2: загрузите 100 точек через upsert (PointStruct с payload=metas[i])
client.upsert(
    "knowledge_base",
    points=[
        models.PointStruct(id=i, vector=vec, payload=meta)
        for i, (vec, meta) in enumerate(zip(corpus_vecs, metas))
    ],
    wait=True,
)
print("Записей в коллекции:", client.count("knowledge_base").count)

# --- Семантический поиск с фильтром по категории ---
QUERY = "сроки и условия возврата средств"
e5 = SentenceTransformer("intfloat/multilingual-e5-base")
q_vec = e5.encode(["query: " + QUERY], normalize_embeddings=True)[0]

# Пропуск 3: поиск топ-3 с query_filter по категории "юридический"
hits = client.query_points(
    "knowledge_base",
    query=q_vec.tolist(),
    limit=3,
    query_filter=models.Filter(must=[                 # Фильтр по метаданным
        models.FieldCondition(
            key="category",
            match=models.MatchValue(value="юридический")
        ),
    ]),
    with_payload=True,
).points

print(f"\nЗапрос: {QUERY!r} | фильтр category='юридический'")
for h in hits:
    p = h.payload
    print(f"  score={h.score:.3f} [{p['category']}/{p['source']}] {texts[h.id]}")

Записей в коллекции: 100

Запрос: 'сроки и условия возврата средств' | фильтр category='юридический'
  score=0.827 [юридический/web] Возврат предоплаты осуществляется в течение десяти рабочих дней.
  score=0.804 [юридический/confluence] Расторжение договора возможно при письменном уведомлении за месяц.
  score=0.798 [юридический/pdf] Гарантийные обязательства действуют двенадцать месяцев с даты продажи.
